# Mukhyamantri Kisan Kalyan Yojana — Approval Prediction Pipeline
### Production-ready XGBoost classifier with GridSearchCV tuning, SHAP explainability, and model persistence

## 1. Import Libraries & Configuration

In [ ]:
# ── Standard library ────────────────────────────────────────────────────────
import os
import json
import warnings
warnings.filterwarnings("ignore")

# ── Data manipulation ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Scikit-learn ─────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# ── XGBoost ──────────────────────────────────────────────────────────────────
from xgboost import XGBClassifier

# ── SHAP ─────────────────────────────────────────────────────────────────────
import shap

# ── Model persistence ────────────────────────────────────────────────────────
import joblib

# ── Global configuration ─────────────────────────────────────────────────────
CFG = {
    "csv_file"   : "mukhyamantri_kisan_kalyan_yojana_dataset.csv",
    "target_col" : "approval_status",
    "test_size"  : 0.20,
    "random_seed": 42,
    "cv_folds"   : 3,
    "model_path" : "kisan_approval_model.pkl",
    "features_path": "kisan_model_features.json",
}

# Consistent plot style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

print("✅ Libraries loaded. Configuration:")
for k, v in CFG.items():
    print(f"   {k:<20} = {v}")

## 2. Load Dataset

In [ ]:
def load_data(filepath: str) -> pd.DataFrame:
    """
    Load CSV dataset from *filepath*.
    Raises FileNotFoundError with a clear message if the file is missing.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(
            f"Dataset not found at '{filepath}'.\n"
            "Place the CSV in the same directory as this notebook."
        )
    df = pd.read_csv(filepath)
    print(f"✅ Dataset loaded  →  {df.shape[0]:,} rows  ×  {df.shape[1]} columns")
    return df


# ── Load ─────────────────────────────────────────────────────────────────────
df = load_data(CFG["csv_file"])

print("\n── Column names ─────────────────────────────────────────────────────")
print(df.columns.tolist())

print("\n── First 5 rows ─────────────────────────────────────────────────────")
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
def run_eda(df: pd.DataFrame, target_col: str) -> None:
    """Print shape, class balance, and missing-value report."""

    sep = "─" * 60

    # 1. Shape
    print(f"{sep}\n📐 Dataset Shape\n{sep}")
    print(f"   Rows    : {df.shape[0]:>8,}")
    print(f"   Columns : {df.shape[1]:>8,}")
    print(f"   Features: {df.shape[1] - 1:>8,}  (excluding target)")

    # 2. Class balance
    print(f"\n{sep}\n⚖️  Class Balance  —  '{target_col}'\n{sep}")
    counts = df[target_col].value_counts()
    pcts   = df[target_col].value_counts(normalize=True) * 100
    balance_df = pd.DataFrame({"Count": counts, "Percentage (%)": pcts.round(2)})
    print(balance_df.to_string())

    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(counts.index.astype(str), counts.values,
           color=["#4C72B0", "#DD8452"], edgecolor="white", width=0.5)
    ax.set_title("Class Distribution", fontweight="bold")
    ax.set_xlabel("approval_status  (0 = Rejected, 1 = Approved)")
    ax.set_ylabel("Count")
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 200,
                f'{bar.get_height():,.0f}', ha='center', fontsize=9)
    plt.tight_layout()
    plt.show()

    # 3. Missing values
    print(f"\n{sep}\n🔍 Missing Values\n{sep}")
    missing = df.isnull().sum()
    missing_any = missing[missing > 0]
    if missing_any.empty:
        print("   ✅ No missing values found.")
    else:
        print(missing_any.to_string())

    # 4. Data types summary
    print(f"\n{sep}\n📊 Data Types & Basic Stats\n{sep}")
    print(df.dtypes.value_counts().to_string())
    print()
    print(df.describe().T[["mean", "std", "min", "max"]].round(2).to_string())


run_eda(df, CFG["target_col"])

## 4. Data Preprocessing & Stratified Split

In [ ]:
def add_realistic_noise(X: pd.DataFrame, seed: int, noise_std: float = 0.15) -> pd.DataFrame:
    """
    Inject Gaussian noise into continuous features to simulate real-world
    data imperfections (measurement error, OCR mistakes, manual entry errors).

    Binary/flag columns (only 0/1 values) are left intact.
    Continuous columns get N(0, noise_std * feature_std) noise added.
    5% of values are randomly flipped (missing-value proxy set to -1).
    """
    rng = np.random.default_rng(seed)
    X_noisy = X.copy().astype(float)

    for col in X_noisy.columns:
        unique_vals = X_noisy[col].nunique()

        if unique_vals > 2:                          # continuous feature
            col_std = X_noisy[col].std()
            noise   = rng.normal(0, noise_std * col_std, size=len(X_noisy))
            X_noisy[col] += noise

            # 3% random missing → mark as -1 (realistic for govt forms)
            missing_mask = rng.random(len(X_noisy)) < 0.03
            X_noisy.loc[missing_mask, col] = -1
        else:                                        # binary flag
            # 4% random bit flip (human entry error)
            flip_mask = rng.random(len(X_noisy)) < 0.04
            X_noisy.loc[flip_mask, col] = 1 - X_noisy.loc[flip_mask, col]

    print(f"   💥 Noise injected  |  noise_std={noise_std}  |  "
          f"continuous cols noised + 3% missing markers  |  binary cols 4% flipped")
    return X_noisy


def preprocess_and_split(df: pd.DataFrame, target_col: str, test_size: float, seed: int):
    """
    Separate features / target, inject realistic noise, and perform a
    stratified 80/20 split.
    Returns: X_train, X_test, y_train, y_test, feature_names
    """
    df = df.dropna(subset=[target_col]).copy()

    X = df.drop(columns=[target_col])
    y = df[target_col].astype(int)

    feature_names = X.columns.tolist()

    # ── Inject noise BEFORE splitting (simulates dirty real-world data) ──────
    X = add_realistic_noise(X, seed=seed, noise_std=0.20)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=seed,
        stratify=y
    )

    print("✅ Data split complete")
    print(f"   Train  : {X_train.shape[0]:>7,} rows  |  class ratio: {y_train.mean():.3f}")
    print(f"   Test   : {X_test.shape[0]:>7,} rows  |  class ratio: {y_test.mean():.3f}")
    print(f"   Features: {len(feature_names)}")

    return X_train, X_test, y_train, y_test, feature_names


# ── Run preprocessing ────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test, feature_names = preprocess_and_split(
    df,
    CFG["target_col"],
    CFG["test_size"],
    CFG["random_seed"]
)

## 5. Train XGBoost Classifier (Baseline)

In [ ]:
def train_xgboost(X_train, y_train, X_test, y_test, seed: int) -> XGBClassifier:
    """
    Train a regularised XGBClassifier that generalises well on noisy data.

    Regularisation choices (prevent overfitting on synthetic-like data):
      max_depth=4        — shallower trees, less memorisation
      min_child_weight=5 — require more samples per leaf
      gamma=1.0          — minimum loss-reduction to split (pruning)
      reg_alpha=0.5      — L1 regularisation on leaf weights
      reg_lambda=2.0     — L2 regularisation on leaf weights
      subsample=0.7      — row subsampling per tree
      colsample_bytree=0.6 — feature subsampling per tree
    """
    neg = (y_train == 0).sum()
    pos = (y_train == 1).sum()
    spw = round(neg / pos, 4)
    print(f"   scale_pos_weight = {spw}  (neg={neg:,} / pos={pos:,})")

    model = XGBClassifier(
        n_estimators          = 400,
        max_depth             = 4,         # shallower → less overfit
        learning_rate         = 0.05,
        min_child_weight      = 5,         # require more samples per leaf
        gamma                 = 1.0,       # min gain to make a split
        subsample             = 0.7,       # row sampling
        colsample_bytree      = 0.6,       # feature sampling
        reg_alpha             = 0.5,       # L1
        reg_lambda            = 2.0,       # L2
        scale_pos_weight      = spw,
        eval_metric           = "logloss",
        early_stopping_rounds = 30,
        random_state          = seed,
        n_jobs                = -1,
        verbosity             = 0,
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    print(f"✅ XGBoost trained  |  best iteration: {model.best_iteration}")
    return model


# ── Train baseline model ─────────────────────────────────────────────────────
baseline_model = train_xgboost(X_train, y_train, X_test, y_test, CFG["random_seed"])

## 6. Hyperparameter Tuning with GridSearchCV

In [ ]:
def tune_hyperparameters(X_train, y_train, seed: int, cv_folds: int):
    """
    GridSearchCV over a param_grid that includes regularisation terms.
    Scoring: ROC-AUC  |  CV: Stratified K-Fold
    Returns: (best_estimator, best_params, cv_results_df)
    """
    neg = (y_train == 0).sum()
    pos = (y_train == 1).sum()
    spw = round(neg / pos, 4)

    base_xgb = XGBClassifier(
        eval_metric       = "logloss",
        scale_pos_weight  = spw,
        random_state      = seed,
        n_jobs            = -1,
        verbosity         = 0,
    )

    param_grid = {
        "n_estimators"   : [100, 200],
        "max_depth"      : [3, 4, 5],
        "learning_rate"  : [0.05, 0.10],
        "min_child_weight": [3, 7],
        "gamma"          : [0.5, 1.5],
        "reg_alpha"      : [0.1, 1.0],   # L1
        "reg_lambda"     : [1.0, 3.0],   # L2
    }

    total_fits = (
        len(param_grid["n_estimators"]) *
        len(param_grid["max_depth"]) *
        len(param_grid["learning_rate"]) *
        len(param_grid["min_child_weight"]) *
        len(param_grid["gamma"]) *
        len(param_grid["reg_alpha"]) *
        len(param_grid["reg_lambda"]) *
        cv_folds
    )
    print(f"🔍 Running GridSearchCV  |  {total_fits} fits  |  scoring = roc_auc")

    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=seed)

    grid_search = GridSearchCV(
        estimator          = base_xgb,
        param_grid         = param_grid,
        scoring            = "roc_auc",
        cv                 = cv,
        n_jobs             = -1,
        verbose            = 1,
        refit              = True,
        return_train_score = False,
    )

    grid_search.fit(X_train, y_train)

    best_params = grid_search.best_params_
    best_score  = grid_search.best_score_

    print(f"\n✅ Best CV ROC-AUC : {best_score:.4f}")
    print("   Best parameters:")
    for k, v in best_params.items():
        print(f"     {k:<22} = {v}")

    cv_df = (
        pd.DataFrame(grid_search.cv_results_)
        .sort_values("mean_test_score", ascending=False)
        .reset_index(drop=True)
    )

    return grid_search.best_estimator_, best_params, cv_df


# ── Tune ─────────────────────────────────────────────────────────────────────
best_model, best_params, cv_results = tune_hyperparameters(
    X_train, y_train,
    CFG["random_seed"],
    CFG["cv_folds"]
)

print("\n── Top 5 parameter combinations ────────────────────────────────────")
cols = ["mean_test_score", "std_test_score",
        "param_n_estimators", "param_max_depth",
        "param_learning_rate", "param_reg_alpha", "param_reg_lambda"]
print(cv_results[cols].head().to_string(index=False))

## 7. Model Evaluation

In [ ]:
def evaluate_model(model, X_test, y_test) -> dict:
    """
    Compute and display a full set of classification metrics.
    Plots a heatmap confusion matrix.
    Returns: dict of all scalar metrics.
    """
    y_pred      = model.predict(X_test)
    y_prob      = model.predict_proba(X_test)[:, 1]

    metrics = {
        "Accuracy" : accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall"   : recall_score(y_test, y_pred, zero_division=0),
        "F1-score" : f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC"  : roc_auc_score(y_test, y_prob),
    }

    sep = "─" * 50
    print(f"{sep}\n📊 Classification Metrics\n{sep}")
    for name, val in metrics.items():
        bar = "█" * int(val * 30)
        print(f"  {name:<12}: {val:.4f}  {bar}")

    print(f"\n{sep}\n📋 Classification Report\n{sep}")
    print(classification_report(y_test, y_pred,
          target_names=["Rejected (0)", "Approved (1)"]))

    # ── Confusion matrix ──────────────────────────────────────────────────────
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Pred: 0", "Pred: 1"],
        yticklabels=["True: 0", "True: 1"],
        linewidths=0.5, ax=ax
    )
    ax.set_title("Confusion Matrix", fontweight="bold")
    ax.set_ylabel("Actual")
    ax.set_xlabel("Predicted")
    plt.tight_layout()
    plt.show()

    return metrics


# ── Evaluate tuned model ─────────────────────────────────────────────────────
eval_metrics = evaluate_model(best_model, X_test, y_test)

## 8. Feature Importance & ROC Curve Plots

In [ ]:
def plot_feature_importance(model, feature_names: list, top_n: int = 15) -> None:
    """Bar chart of the top_n most important features (gain importance)."""
    importances = model.feature_importances_
    feat_df = (
        pd.DataFrame({"Feature": feature_names, "Importance": importances})
        .sort_values("Importance", ascending=False)
        .head(top_n)
        .sort_values("Importance")             # ascending for horizontal bar
    )

    fig, ax = plt.subplots(figsize=(8, 5))
    colors = plt.cm.viridis_r(
        np.linspace(0.2, 0.9, len(feat_df))
    )
    bars = ax.barh(feat_df["Feature"], feat_df["Importance"],
                   color=colors, edgecolor="white")
    ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=8)
    ax.set_title(f"Top {top_n} Feature Importances (XGBoost Gain)",
                 fontweight="bold")
    ax.set_xlabel("Importance Score")
    ax.set_xlim(right=feat_df["Importance"].max() * 1.15)
    plt.tight_layout()
    plt.show()


def plot_roc_curve(model, X_test, y_test) -> None:
    """
    Plot ROC curve and annotate with AUC score.
    AUC = ∫₀¹ TPR d(FPR)
    """
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_score    = roc_auc_score(y_test, y_prob)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, lw=2, color="#4C72B0",
            label=f"XGBoost  (AUC = {auc_score:.4f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1.2, label="Random classifier")
    ax.fill_between(fpr, tpr, alpha=0.08, color="#4C72B0")
    ax.set_title("Receiver Operating Characteristic (ROC) Curve",
                 fontweight="bold")
    ax.set_xlabel("False Positive Rate  (FPR)")
    ax.set_ylabel("True Positive Rate   (TPR)")
    ax.legend(loc="lower right")
    ax.annotate(
        r"$AUC = \int_0^1 TPR\,d(FPR)$",
        xy=(0.55, 0.12), fontsize=9, color="#333"
    )
    plt.tight_layout()
    plt.show()


# ── Plot ─────────────────────────────────────────────────────────────────────
plot_feature_importance(best_model, feature_names, top_n=15)
plot_roc_curve(best_model, X_test, y_test)

## 9. SHAP Explainability

In [ ]:
def run_shap_explainability(model, X_test: pd.DataFrame, sample_idx: int = 0) -> None:
    """
    Global SHAP summary plot  +  single-prediction waterfall plot.

    Parameters
    ----------
    model      : trained XGBClassifier (best_model)
    X_test     : test feature DataFrame
    sample_idx : index of the sample to explain individually
    """
    print("⏳ Computing SHAP values (TreeExplainer — fast for XGBoost) …")
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)   # shape: (n_samples, n_features)

    # ── Global summary — beeswarm ─────────────────────────────────────────────
    print("\n📊 Global SHAP Feature Importance (beeswarm plot)")
    plt.figure()
    shap.summary_plot(
        shap_values,
        X_test,
        plot_type="dot",          # beeswarm
        max_display=15,
        show=True
    )

    # ── Global summary — bar ──────────────────────────────────────────────────
    print("\n📊 Global Mean |SHAP| (bar plot)")
    plt.figure()
    shap.summary_plot(
        shap_values,
        X_test,
        plot_type="bar",
        max_display=15,
        show=True
    )

    # ── Single prediction — waterfall ─────────────────────────────────────────
    print(f"\n🔍 Single Prediction Explanation  (test row index = {sample_idx})")
    single_shap = shap.Explanation(
        values          = shap_values[sample_idx],
        base_values     = explainer.expected_value,
        data            = X_test.iloc[sample_idx].values,
        feature_names   = X_test.columns.tolist(),
    )
    plt.figure()
    shap.plots.waterfall(single_shap, show=True)

    y_pred_single = model.predict(X_test.iloc[[sample_idx]])[0]
    y_prob_single = model.predict_proba(X_test.iloc[[sample_idx]])[0, 1]
    label = "Approved ✅" if y_pred_single == 1 else "Rejected ❌"
    print(f"\n   Prediction for row {sample_idx}: {label}  "
          f"(probability = {y_prob_single:.4f})")


# ── Run SHAP ─────────────────────────────────────────────────────────────────
run_shap_explainability(best_model, X_test, sample_idx=0)

## 10. Save Model & Feature List

In [ ]:
def save_artifacts(model, feature_names: list, model_path: str, features_path: str) -> None:
    """
    Persist the trained model (joblib) and feature list (JSON).

    Parameters
    ----------
    model         : final trained XGBClassifier
    feature_names : list of feature column names used during training
    model_path    : destination .pkl file path
    features_path : destination .json file path
    """
    # 1. Save model
    joblib.dump(model, model_path)
    model_size_kb = os.path.getsize(model_path) / 1024
    print(f"✅ Model saved  →  {model_path}  ({model_size_kb:.1f} KB)")

    # 2. Save feature list with metadata
    artifact = {
        "feature_names" : feature_names,
        "n_features"    : len(feature_names),
        "best_params"   : best_params,
        "eval_metrics"  : {k: round(float(v), 6) for k, v in eval_metrics.items()},
        "model_path"    : model_path,
    }
    with open(features_path, "w") as f:
        json.dump(artifact, f, indent=2)
    print(f"✅ Feature list saved  →  {features_path}")

    # 3. Verify round-trip
    loaded_model    = joblib.load(model_path)
    verify_pred     = loaded_model.predict_proba(X_test[:5])[:, 1]
    print(f"\n🔁 Round-trip verification (first 5 probabilities):")
    print("  ", np.round(verify_pred, 4))
    print("\n── Saved Artifacts Summary ──────────────────────────────────────────")
    print(f"  Model file     : {model_path}")
    print(f"  Features file  : {features_path}")
    print(f"  Feature count  : {len(feature_names)}")
    print(f"  Best ROC-AUC   : {eval_metrics['ROC-AUC']:.4f}")
    print(f"  Best Accuracy  : {eval_metrics['Accuracy']:.4f}")
    print(f"  Best params    : {best_params}")


# ── Save ─────────────────────────────────────────────────────────────────────
save_artifacts(
    best_model,
    feature_names,
    CFG["model_path"],
    CFG["features_path"]
)